# Stage 4: Faithfulness-Constrained Iterative Simplification
**FIXED VERSION** — 3 bugs corrected (see comments marked FIX).

In [ ]:
!pip install -q transformers datasets accelerate rouge-score nltk tqdm

import os, json, re, time, random, logging
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from collections import Counter

import torch
import numpy as np
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(levelname)s  %(message)s")
log = logging.getLogger(__name__)

# ── Paths ──────────────────────────────────────────────────────────────────────
STAGE3_OUTPUT_DIR     = "/kaggle/input/datasets/saarthaksabharwal/stage3-output"
STAGE3_RESULTS_FILE   = f"{STAGE3_OUTPUT_DIR}/simplified_test_clauses.json"
BART_MODEL_DIR        = f"{STAGE3_OUTPUT_DIR}/final_bart_model"

TEST_CONSTRAINTS      = "/kaggle/input/datasets/harshitgoyal41/stage2op/stage2_output/test_constraints.json"

OUTPUT_DIR            = "/kaggle/working/stage4_output"

DEBERTA_BASE_MODEL    = "cross-encoder/nli-deberta-v3-large"
DEBERTA_OUTPUT_DIR    = f"{OUTPUT_DIR}/deberta_contractnli"

CONTRACT_NLI_PATH     = "/kaggle/input/contractnli/contract_nli_train.jsonl"

# ── Entailment gate hyperparameters ───────────────────────────────────────────
ENTAILMENT_THRESHOLD  = 0.85
MAX_RETRIES           = 3

# ── BART generation hyperparameters ──────────────────────────────────────────
MAX_INPUT_LEN         = 512
MAX_TARGET_LEN        = 256
BART_NUM_BEAMS        = 4

# ── DeBERTa fine-tuning hyperparameters ──────────────────────────────────────
NLI_EPOCHS            = 2
NLI_BATCH_SIZE        = 16
NLI_LR                = 2e-5
NLI_MAX_LEN           = 512

# ── Misc ──────────────────────────────────────────────────────────────────────
SEED                  = 42
USE_FP16              = True

# ── FIX 3: minimum clause length to skip trivial headers ──────────────────────
MIN_CLAUSE_LEN        = 50

os.makedirs(OUTPUT_DIR, exist_ok=True)
torch.manual_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"BART model dir: {BART_MODEL_DIR}")
print(f"DeBERTa base: {DEBERTA_BASE_MODEL}")

### Stage 3 Utility Functions

In [ ]:
ENTITY_MARKERS = {
    "DATE":         ("<DATE>",  "</DATE>"),
    "MONEY":        ("<MONEY>", "</MONEY>"),
    "DURATION":     ("<DUR>",   "</DUR>"),
    "LEGAL_REF":    ("<LREF>",  "</LREF>"),
    "PARTY":        ("<PARTY>", "</PARTY>"),
    "JURISDICTION": ("<JURIS>", "</JURIS>"),
}

OBLIGATION_TOKENS = [
    "[MANDATORY]", "[PROHIBITIVE]", "[PERMISSIVE]",
    "[CONDITIONAL]", "[DECLARATIVE]",
]

DATE_PATTERNS = [
    re.compile(
        r"\b(?:January|February|March|April|May|June|July|August|September|"
        r"October|November|December)\s+\d{1,2},?\s+\d{4}\b", re.IGNORECASE),
    re.compile(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b"),
    re.compile(r"\b\d{4}[-/]\d{2}[-/]\d{2}\b"),
]
MONEY_PATTERNS = [
    re.compile(r"\$[\d,]+(?:\.\d{1,2})?(?:\s*(?:million|billion|thousand))?", re.IGNORECASE),
    re.compile(r"\b(?:USD|EUR|GBP|CAD)\s*[\d,]+(?:\.\d{1,2})?", re.IGNORECASE),
    re.compile(r"\b\d+(?:\.\d+)?%\b"),
]
DURATION_PATTERNS = [
    re.compile(r"\b\d+\s*(?:days?|months?|years?|weeks?|calendar\s+days?|business\s+days?)\b", re.IGNORECASE),
]
LEGAL_REF_PATTERNS = [
    re.compile(r"\bSection\s+\d+(?:\.\d+)*(?:\([a-z]\))?", re.IGNORECASE),
    re.compile(r"\bArticle\s+(?:\d+|[IVXLC]+)\b", re.IGNORECASE),
]

OBLIGATION_KEYWORDS = {
    "MANDATORY":   ["shall", "must", "is required to", "will", "agrees to"],
    "PROHIBITIVE": ["shall not", "must not", "may not", "will not", "cannot"],
    "PERMISSIVE":  ["may", "can", "is entitled to", "has the right to"],
    "CONDITIONAL": ["if", "provided that", "subject to", "unless", "notwithstanding"],
    "DECLARATIVE": ["means", "shall mean", "is defined as", "refers to"],
}

def extract_entities_regex(text: str) -> List[Dict]:
    spans = []
    for patterns, etype in [
        (DATE_PATTERNS, "DATE"), (MONEY_PATTERNS, "MONEY"),
        (DURATION_PATTERNS, "DURATION"), (LEGAL_REF_PATTERNS, "LEGAL_REF"),
    ]:
        for pat in patterns:
            for m in pat.finditer(text):
                spans.append({"type": etype, "text": m.group(),
                               "start": m.start(), "end": m.end()})
    spans.sort(key=lambda x: (x["start"], -x["end"]))
    filtered, last_end = [], -1
    for e in spans:
        if e["start"] >= last_end:
            filtered.append(e)
            last_end = e["end"]
    return filtered

def detect_obligation(text: str) -> str:
    tl = text.lower()
    for obl in ["PROHIBITIVE", "PERMISSIVE", "CONDITIONAL", "DECLARATIVE", "MANDATORY"]:
        if any(kw in tl for kw in OBLIGATION_KEYWORDS[obl]):
            return obl
    return "DECLARATIVE"

def mark_entities_in_text(text: str, entities: List[Dict]) -> str:
    if not entities:
        return text
    for e in sorted(entities, key=lambda x: x.get("start", 0), reverse=True):
        s, end_, etype = e.get("start"), e.get("end"), e.get("type", "")
        if s is not None and end_ is not None and etype in ENTITY_MARKERS:
            ot, ct = ENTITY_MARKERS[etype]
            text = text[:s] + ot + text[s:end_] + ct + text[end_:]
    return text

print("Stage 3 utility functions loaded.")

### Load Data & BART Checkpoint

In [ ]:
print("\nLoading Stage 3 simplified clauses...")
with open(STAGE3_RESULTS_FILE, "r", encoding="utf-8") as f:
    stage3_results = json.load(f)

mutable_results   = [r for r in stage3_results if not r.get("immutable", False)]
immutable_results = [r for r in stage3_results if r.get("immutable", False)]

print(f"  Total clauses:    {len(stage3_results)}")
print(f"  Mutable (active): {len(mutable_results)}")
print(f"  Immutable (skip): {len(immutable_results)}")

def load_constraint_lookup(constraint_path: str) -> Dict:
    lookup = {}
    if os.path.exists(constraint_path):
        with open(constraint_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        for doc_entry in data.get("documents", []):
            doc_id = doc_entry["doc_id"]
            for clause in doc_entry.get("clauses", []):
                lookup[(doc_id, clause["clause_index"])] = clause
        print(f"  Loaded {len(lookup)} constraint profiles from {Path(constraint_path).name}")
    else:
        print(f"  WARNING: {constraint_path} not found — will use regex fallback")
    return lookup

print("\nLoading Stage 2 test constraints...")
test_constraint_lookup = load_constraint_lookup(TEST_CONSTRAINTS)

from transformers import BartForConditionalGeneration, BartTokenizer

print(f"\nLoading BART from {BART_MODEL_DIR} ...")
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL_DIR)
bart_model = BartForConditionalGeneration.from_pretrained(BART_MODEL_DIR).to(device)
bart_model.eval()
print(f"  BART vocabulary: {len(bart_tokenizer):,}")
print(f"  BART parameters: {sum(p.numel() for p in bart_model.parameters()):,}")

def bart_generate(input_text: str) -> str:
    enc = bart_tokenizer(
        input_text,
        max_length=MAX_INPUT_LEN,
        truncation=True,
        return_tensors="pt",
    ).to(device)
    use_fp16 = USE_FP16 and device.type == "cuda"
    with torch.no_grad():
        with torch.amp.autocast("cuda", enabled=use_fp16):
            ids = bart_model.generate(
                **enc,
                max_length=MAX_TARGET_LEN,
                num_beams=BART_NUM_BEAMS,
                length_penalty=1.0,
                early_stopping=True,
                no_repeat_ngram_size=3,
            )
    return bart_tokenizer.decode(ids[0], skip_special_tokens=True)

### Load & Fine-tune DeBERTa NLI Model

In [ ]:
from transformers import (
    AutoTokenizer as AutoTok,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

# FIX 1: Do NOT pass id2label / label2id overrides.
NLI_LABEL2ID = {"entailment": 0, "neutral": 1, "contradiction": 2}
NLI_ID2LABEL = {v: k for k, v in NLI_LABEL2ID.items()}

class NLIDataset(Dataset):
    def __init__(self, records: List[Dict], tokenizer, max_len: int = 512):
        self.records   = records
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]
        enc = self.tokenizer(
            r["premise"], r["hypothesis"],
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(),
            "attention_mask": enc["attention_mask"].squeeze(),
            "labels":         torch.tensor(nli_model.config.label2id[r["label"]], dtype=torch.long),
        }

print(f"\nLoading DeBERTa NLI model: {DEBERTA_BASE_MODEL}")
nli_tokenizer = AutoTok.from_pretrained(DEBERTA_BASE_MODEL)
nli_model = AutoModelForSequenceClassification.from_pretrained(DEBERTA_BASE_MODEL, num_labels=3).to(device)

if os.path.exists(CONTRACT_NLI_PATH):
    print(f"\nFine-tuning DeBERTa on ContractNLI: {CONTRACT_NLI_PATH}")
    contract_records = []
    with open(CONTRACT_NLI_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                obj   = json.loads(line)
                label = obj.get("label", "").lower()
                if label in nli_model.config.label2id:
                    contract_records.append({
                        "premise":    obj["premise"],
                        "hypothesis": obj["hypothesis"],
                        "label":      label,
                    })

    random.shuffle(contract_records)
    n_val             = max(1, int(len(contract_records) * 0.1))
    nli_train_records = contract_records[n_val:]
    nli_val_records   = contract_records[:n_val]

    nli_train_ds = NLIDataset(nli_train_records, nli_tokenizer, NLI_MAX_LEN)
    nli_val_ds   = NLIDataset(nli_val_records,   nli_tokenizer, NLI_MAX_LEN)
    nli_train_dl = DataLoader(nli_train_ds, batch_size=NLI_BATCH_SIZE, shuffle=True,  num_workers=2)
    nli_val_dl   = DataLoader(nli_val_ds,   batch_size=NLI_BATCH_SIZE, shuffle=False, num_workers=2)

    nli_optimizer = AdamW(nli_model.parameters(), lr=NLI_LR)
    total_steps   = len(nli_train_dl) * NLI_EPOCHS
    nli_scheduler = get_linear_schedule_with_warmup(
        nli_optimizer,
        num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps,
    )
    use_fp16 = USE_FP16 and device.type == "cuda"
    scaler   = torch.amp.GradScaler("cuda", enabled=use_fp16)

    best_nli_val_loss = float("inf")
    for epoch in range(1, NLI_EPOCHS + 1):
        nli_model.train()
        total_loss = 0.0
        for batch in tqdm(nli_train_dl, desc=f"NLI Epoch {epoch}/{NLI_EPOCHS} [Train]"):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.amp.autocast("cuda", enabled=use_fp16):
                loss = nli_model(**batch).loss
            scaler.scale(loss).backward()
            scaler.unscale_(nli_optimizer)
            torch.nn.utils.clip_grad_norm_(nli_model.parameters(), 1.0)
            scaler.step(nli_optimizer)
            scaler.update()
            nli_optimizer.zero_grad()
            nli_scheduler.step()
            total_loss += loss.item()

        avg_train = total_loss / len(nli_train_dl)

        nli_model.eval()
        val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for batch in tqdm(nli_val_dl, desc=f"NLI Epoch {epoch}/{NLI_EPOCHS} [Val]", leave=False):
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.amp.autocast("cuda", enabled=use_fp16):
                    out = nli_model(**batch)
                val_loss += out.loss.item()
                preds     = out.logits.argmax(dim=-1)
                correct  += (preds == batch["labels"]).sum().item()
                total    += batch["labels"].size(0)

        avg_val = val_loss / max(len(nli_val_dl), 1)
        acc     = correct  / max(total, 1)
        print(f"  Epoch {epoch} | Train Loss={avg_train:.4f} | Val Loss={avg_val:.4f} | Acc={acc:.4f}")

        if avg_val < best_nli_val_loss:
            best_nli_val_loss = avg_val
            nli_model.save_pretrained(DEBERTA_OUTPUT_DIR)
            nli_tokenizer.save_pretrained(DEBERTA_OUTPUT_DIR)
            print(f"    >>> Best NLI model saved")
else:
    print("ContractNLI not found. Using pre-trained weights.")
    nli_model.eval()

_id2label = nli_model.config.id2label
ENTAIL_IDX = next((idx for idx, lbl in _id2label.items() if "entail" in lbl.lower()), 1)

@torch.no_grad()
def entailment_prob(premise: str, hypothesis: str) -> float:
    enc = nli_tokenizer(premise, hypothesis, max_length=NLI_MAX_LEN, truncation=True, return_tensors="pt").to(device)
    use_fp16 = USE_FP16 and device.type == "cuda"
    with torch.amp.autocast("cuda", enabled=use_fp16):
        logits = nli_model(**enc).logits
    probs = torch.softmax(logits, dim=-1).squeeze()
    return float(probs[ENTAIL_IDX].item())

_test_prob = entailment_prob("The Licensee shall pay a royalty of $10,000 per year.", "The Licensee must pay $10,000 each year.")
print(f"Sanity check — entailment prob: {_test_prob:.4f}")

### Iterative Faithfulness Loop

In [ ]:
FAITHFULNESS_CORRECTION_PREFIX = "rewrite more faithfully"

stage4_results     = []
human_review_flags = []
entail_scores_s3   = []
entail_scores_s4   = []
retry_distribution = Counter()
trivial_skipped    = 0

for record in tqdm(mutable_results, desc="Stage 4 faithfulness loop"):
    doc_id     = record["doc_id"]
    clause_idx = record["clause_idx"]
    original   = record["original"]
    obligation = record.get("obligation", "DECLARATIVE")

    is_trivial = (
        len(original.strip()) < MIN_CLAUSE_LEN
        or original.strip().replace(" ", "").replace("\n", "").isupper()
    )
    if is_trivial:
        trivial_skipped += 1
        stage4_results.append({
            **record, "stage3_simplified": record["simplified"], "stage4_simplified": record["simplified"],
            "stage3_entail_score": None, "stage4_entail_score": None, "retries_used": 0,
            "accepted": True, "flagged_human": False, "method": "skipped_trivial",
        })
        continue

    constraint = test_constraint_lookup.get((doc_id, clause_idx))
    entities = constraint.get("entities", []) if constraint else extract_entities_regex(original)

    current_simplified = record["simplified"]
    s3_score = entailment_prob(original, current_simplified)
    entail_scores_s3.append(s3_score)

    marked_original = mark_entities_in_text(original, entities)
    accepted        = s3_score >= ENTAILMENT_THRESHOLD
    retries_used    = 0
    scores_per_iter = [s3_score]

    for k in range(1, MAX_RETRIES + 1):
        if accepted: break

        bart_input = f"[FAITHFULNESS CORRECTION k={k}] simplify [{obligation}] preserving all facts: {marked_original}"
        rewritten = bart_generate(bart_input)
        new_score = entailment_prob(original, rewritten)
        scores_per_iter.append(new_score)
        retries_used = k

        if new_score >= ENTAILMENT_THRESHOLD:
            current_simplified = rewritten
            accepted = True
        elif new_score > scores_per_iter[-2]:
            current_simplified = rewritten

    final_score = entailment_prob(original, current_simplified)
    entail_scores_s4.append(final_score)
    retry_distribution[retries_used] += 1

    entity_texts = [e["text"] for e in entities]
    missing_ents = [et for et in entity_texts if et not in current_simplified]

    result_record = {
        "doc_id": doc_id, "clause_idx": clause_idx, "original": original,
        "stage3_simplified": record["simplified"], "stage4_simplified": current_simplified,
        "obligation": obligation, "entities": entity_texts, "missing_entities": missing_ents,
        "entities_preserved": len(missing_ents) == 0, "stage3_entail_score": round(s3_score, 4),
        "stage4_entail_score": round(final_score, 4), "entail_scores_per_iter": [round(s, 4) for s in scores_per_iter],
        "retries_used": retries_used, "accepted": accepted, "flagged_human": not accepted,
        "method": "stage4_iterative" if retries_used > 0 else "stage3_passthrough",
    }
    stage4_results.append(result_record)
    if not accepted: human_review_flags.append(result_record)

for r in immutable_results:
    stage4_results.append({
        **r, "stage3_simplified": r["simplified"], "stage4_simplified": r["simplified"],
        "stage3_entail_score": 1.0, "stage4_entail_score": 1.0, "retries_used": 0,
        "accepted": True, "flagged_human": False, "method": "skipped_immutable",
    })

with open(Path(OUTPUT_DIR) / "stage4_simplified_clauses.json", "w", encoding="utf-8") as f:
    json.dump(stage4_results, f, indent=2, ensure_ascii=False)

with open(Path(OUTPUT_DIR) / "human_review_queue.json", "w", encoding="utf-8") as f:
    json.dump(human_review_flags, f, indent=2, ensure_ascii=False)

### Evaluation & Outputs

In [ ]:
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
from rouge_score import rouge_scorer as _rouge_scorer

def count_syllables(word: str) -> int:
    word = word.lower()
    count, vowels = 0, "aeiouy"
    if word and word[0] in vowels: count += 1
    for i in range(1, len(word)):
        if word[i] in vowels and word[i - 1] not in vowels: count += 1
    if word.endswith("e"): count -= 1
    return max(count, 1)

def flesch_kincaid_grade(text: str) -> float:
    sents = nltk.sent_tokenize(text)
    words = [w for w in nltk.word_tokenize(text) if w.isalpha()]
    if not sents or not words: return 0.0
    syllables = sum(count_syllables(w) for w in words)
    return round(0.39 * (len(words) / len(sents)) + 11.8 * (syllables / len(words)) - 15.59, 2)

scorer = _rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

active = [r for r in stage4_results if r.get("method") not in ("skipped_immutable", "skipped_trivial")]

rouge1_s3, rouge1_s4 = [], []
scored_s3 = [r["stage3_entail_score"] for r in active if r.get("stage3_entail_score") is not None]
scored_s4 = [r["stage4_entail_score"] for r in active if r.get("stage4_entail_score") is not None]

for r in active:
    rouge1_s3.append(scorer.score(r["original"], r["stage3_simplified"])["rouge1"].fmeasure)
    rouge1_s4.append(scorer.score(r["original"], r["stage4_simplified"])["rouge1"].fmeasure)

eval_metrics = {
    "entailment_gate": {
        "stage3_avg_score": round(float(np.mean(scored_s3)), 4),
        "stage4_avg_score": round(float(np.mean(scored_s4)), 4),
    },
    "rouge1": {
        "stage3_avg": round(float(np.mean(rouge1_s3)), 4),
        "stage4_avg": round(float(np.mean(rouge1_s4)), 4),
    }
}

with open(Path(OUTPUT_DIR) / "stage4_evaluation_metrics.json", "w") as f:
    json.dump(eval_metrics, f, indent=2)

print("Evaluation complete. Metrics saved.")

corrected = [r for r in stage4_results if r.get("retries_used", 0) > 0]
for r in corrected[:2]:
    print(f"\nDoc: {r['doc_id']}, Clause: {r['clause_idx']}")
    print(f"  ORIGINAL: {r['original'][:100]}...")
    print(f"  STAGE 4:  {r['stage4_simplified'][:100]}...")